# The Repeat Purchase Lab: a customer analytics walkthrough

This notebook is a companion to `run_project.py`. It shows the main tables and a few checks in plain Python. The included CSV is synthetic demo data; rerun the project with the original UCI workbook before presenting numbers as findings.

**Business question:** which customer relationships should a retention team protect, grow or try to win back?

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "outputs").exists() and (ROOT.parent / "outputs").exists():
    ROOT = ROOT.parent
metrics = json.loads((ROOT / "outputs" / "metrics.json").read_text(encoding="utf-8"))
print(metrics["source_label"])
print(f"Period: {metrics['data_start']} to {metrics['data_end']}")
print(f"Known customers: {metrics['known_customers']:,}")
print(f"Repeat customer share: {metrics['repeat_customer_rate_pct']:.1f}%")

## Start with a customer, not a transaction line

The analysis counts distinct invoices as purchases. A basket with five products is still one purchase event. This matters because order frequency is part of RFM segmentation.

In [ ]:
customers = pd.read_csv(ROOT / "outputs" / "customer_metrics.csv")
segments = pd.read_csv(ROOT / "outputs" / "segment_summary.csv")
display(segments.sort_values("historic_value_gbp", ascending=False))

## Value can be concentrated

The chart ranks customers by their historic purchase value and shows the cumulative share of value. This is a descriptive view of the observation window. It is not a profit curve.

In [ ]:
curve = pd.read_csv(ROOT / "outputs" / "customer_value_concentration.csv")
plt.figure(figsize=(8, 5))
plt.plot(curve["customer_share_pct"], curve["cumulative_value_share_pct"], label="Observed customer value")
plt.plot([0, 100], [0, 100], linestyle="--", label="Even distribution")
plt.xlabel("Cumulative customer share (%)")
plt.ylabel("Cumulative historic value (%)")
plt.title("Customer value concentration")
plt.legend(frameon=False)
plt.show()

## Retention takes time to develop

Cohorts are grouped by first-purchase month. Empty cells near the right edge mean that later cohort ages have not been observed in the file. They are not zero retention.

In [ ]:
cohorts = pd.read_csv(ROOT / "outputs" / "cohort_retention.csv")
cohorts.head(12)

## Repeat-purchase score: keep the claim narrow

The model uses only pre-snapshot history and predicts whether another purchase occurred during the next 90 days. The customer holdout helps check whether the ranking generalises to customers not used to fit the model. It does not prove a campaign will work and is not a full lifetime-value model.

In [ ]:
model_metrics = metrics.get("repeat_model", {})
model_metrics

## Business follow-through

A score is not an action plan. Before spending money, choose one segment, state the hypothesis, hold out a comparable group, and measure incremental repeat orders and contribution after contact cost. The source does not include product margins, acquisition costs or campaign costs, so any margin and outreach assumptions must be supplied for the scenario planner.